In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import json
import re
from deep_translator import GoogleTranslator
from selenium.webdriver.common.keys import Keys

In [6]:
def to_json(data):
    """
    This function takes list of dictionary as Input and 
    then Creates a JSON file in which Input data is stored
    """
    with open("data_dict.json", "w") as outfile:
        json.dump(data, outfile,indent=4)
        outfile.close()

In [7]:
def get_data(slug_name):    
    data_list = []
    url = "https://www.state.gov/narcotics-rewards-program/target-information/wanted/"
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized") 
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--log-level=3")
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    translator = GoogleTranslator(target='english')
    driver = webdriver.Chrome(options=options)
    driver.maximize_window()
    driver.get(url)
    list1 = driver.find_elements(By.XPATH, f'/html/body/div[1]/div[2]/main/article/div/div[2]/div/ul/li/a')
    for i in range(1, len(list1)+1):
        link = driver.find_element(By.XPATH, f'/html/body/div[1]/div[2]/main/article/div/div[2]/div/ul/li[{i}]/a').get_attribute("href")
        driver.execute_script("window.open('');")
        driver.switch_to.window(driver.window_handles[1])
        driver.get(link)
        try:
            search = driver.find_element(By.XPATH, f'/html/body/div[1]/div[2]/main/article/div/div[2]/div/div/div[2]/form/div[1]/select')
            search.send_keys("30")
            search.send_keys(Keys.RETURN)
        except:
            pass
        list2 = driver.find_elements(By.XPATH, f'/html/body/div[1]/div[2]/main/article/div/div[2]/div/ul/li/a')
    #     print(len(list2))
        for j in range(1, len(list2)+1):
            image = ""
            driver.find_element(By.XPATH, f'/html/body/div[1]/div[2]/main/article/div/div[2]/div/ul/li[{j}]/a').click()
            data_dict = {}
            status = "Wanted"
            description =""
            reward = ""
            charges =""
            identifierType = ""
            distinguishMarks = ""
            identifierID = ""
            fullName = driver.find_element(By.XPATH, f'/html/body/div[1]/div[2]/main/article/section/div[2]/div/h1').text
            fullName = fullName.replace("– New Target", "").replace("— New Target", "")
            print(fullName)
            try:
                image = driver.find_element(By.XPATH, f'/html/body/div[1]/div[2]/main/article/div[2]/div/figure/strong/img').get_attribute('src')
            except:
                try:
                    image = driver.find_element(By.XPATH, f'/html/body/div[1]/div[2]/main/article/div[2]/div/figure/b/img').get_attribute('src')
                except:
                    try:
                        image = driver.find_element(By.XPATH, f'/html/body/div[1]/div[2]/main/article/div[2]/div/figure/img').get_attribute('src')
                    except:
                        try:
                            image = driver.find_element(By.XPATH, f'/html/body/div[1]/div[2]/main/article/div[2]/div/p[2]/span/img').get_attribute('src')
                        except:
                            pass

            print(image)
            try:
                driver.find_element(By.XPATH, f'/html/body/div[5]/div/div/button').click()
            except:
                pass
            try:
                lastUpdatedAt = driver.find_element(By.XPATH, f'/html/body/div[1]/div[2]/main/article/section/div[2]/div/div[1]/p[3]').text
            except:
                pass
            bioInfo = driver.find_element(By.XPATH, f'/html/body/div[1]/div[2]/main/article/div[2]/div/p[4]').text
            if "NAME" not in bioInfo:
                bioInfo = driver.find_element(By.XPATH, f'/html/body/div[1]/div[2]/main/article/div[2]/div/p[2]').text
                if "NAME" not in bioInfo:
                    bioInfo = driver.find_element(By.XPATH, f'/html/body/div[1]/div[2]/main/article/div[2]/div/p[1]').text
                    if "NAME" not in bioInfo:
                        bioInfo = driver.find_element(By.XPATH, f'/html/body/div[1]/div[2]/main/article/div[2]/div/p[3]').text
            try:
                alias = bioInfo.split("ALIASES:")[1].split("\n")[0].replace("\n", "").strip().replace("N/A", "").replace("Unknown", "")
            except:
                alias = driver.find_element(By.XPATH, f'/html/body/div[1]/div[2]/main/article/div[2]/div/p[2]/span[3]').text.strip().replace("N/A", "").replace("Unknown", "")
            print(alias)
            dob = bioInfo.split("DOB:")[1].split("POB:")[0].replace("\n", "").strip().replace("UNKNOWN", "").replace("Unknown", "")
            print(dob)
            placeOfBirthCity = bioInfo.split("POB:")[1].split("NATIONALITY:")[0].replace("\n", "").strip().replace("UNKNOWN", "").replace("Unknown", "")
            print(placeOfBirthCity)
            nationality = bioInfo.split("NATIONALITY:")[1].split("CITIZENSHIP:")[0].replace("\n", "").strip().replace("UNKNOWN", "").replace("Unknown", "")
            print(nationality)
            height = bioInfo.split("HEIGHT:")[1].split("WEIGHT:")[0].replace("\n", "").strip().replace("UNKNOWN", "").replace("Unknown", "")
            print(height)
            weight = bioInfo.split("WEIGHT:")[1].split("HAIR")[0].replace("\n", "").strip().replace("UNKNOWN", "").replace("Unknown", "")
            print(weight)
            hair = bioInfo.split("HAIR COLOR:")[1].split("EYE COLOR:")[0].replace("\n", "").strip().replace("UNKNOWN", "").replace("Unknown", "")
            print(hair)
            eyes = bioInfo.split("EYE COLOR:")[1].split("\n")[0].strip().replace("UNKNOWN", "").replace("Unknown", "")
            print(eyes)
            if fullName == "Luciano Marín Arango":
                identifierType = "COLOMBIAN CEDULA"
                identifierID = bioInfo.split("COLOMBIAN CEDULA:")[1].split("MARKS")[0].strip()
            if "MARKS" in bioInfo:
                distinguishMarks = bioInfo.split("MARKS:")[1].split("\n")[0].strip().replace("UNKNOWN", "").replace("Unknown", "")
            list3 = driver.find_elements(By.XPATH, f'/html/body/div[1]/div[2]/main/article/div[2]/div/p')
            for k in range(1, len(list3)+1):
                info = driver.find_element(By.XPATH, f'/html/body/div[1]/div[2]/main/article/div[2]/div/p[{k}]').text
                if "[Wanted" in info:
                    info = ""
                elif "Poster" in info:
                    info = ""
                elif "NAME" in info:
                    info = ""
                elif "ALL IDENTITIES" in info:
                    info = ""
                elif "charge" in info:
                    charges = info
                elif fullName in info:
                    description = description + info
                else:
                    description = description + info

            description = description.replace("\n", "")
            try:
                reward = description.split("reward of")[1].split("for", 1)[0].strip()
            except:
                try:
                    reward = description.split("REWARD OF")[1].split("for", 1)[0].strip()
                except:
                    pass
            reward = reward.title()
            reward = reward[0:reward.index("Million")+7]
            print(reward)
            print(charges)
            print(description)
            summary = fullName + " is one of the Wanted: Narcotics Rewards Program Targets and has a reward of: " + reward + " for any information provided."
            if fullName:
                data_dict['fullName'] = fullName
            if identifierID:
                data_dict['identifierID'] = identifierID
            if identifierType:
                data_dict['identifierType'] = identifierType
            if status:
                data_dict['status'] = status
            if image:
                data_dict['image'] = image
            if alias:
                data_dict['alias'] = alias
            if dob:
                data_dict['dob'] = dob
            if placeOfBirthCity:
                data_dict['placeOfBirthCity'] = placeOfBirthCity
            if nationality:
                data_dict['nationality'] = nationality
            if height:
                data_dict['height'] = height
            if weight:
                data_dict['weight'] = weight
            if hair:
                data_dict['hair'] = hair
            if eyes:
                data_dict['eyes'] = eyes
            if distinguishMarks:
                data_dict['distinguishMarks'] = distinguishMarks
            if charges:
                data_dict['charges'] = charges
            if description:
                data_dict['description'] = description
            if reward:
                data_dict['reward'] = reward
            if lastUpdatedAt:
                data_dict['lastUpdatedAt'] = lastUpdatedAt
            if summary:
                data_dict['summary'] = summary
            data_list.append(data_dict)
            driver.back()
        try:
            driver.find_element(By.XPATH, f'/html/body/div[5]/div/div/button').click()
        except:
            pass
        driver.close()
        driver.switch_to.window(driver.window_handles[0])
    driver.quit()
    return data_list

In [8]:
if __name__ == '__main__':
    data_list = get_data("add_slug_name")
    to_json(data_list)

Jesus Alfredo Itriago 
https://www.state.gov/wp-content/uploads/2020/09/Jesus-Alfredo-Itriago.jpg
Arnold
January 7, 1958
Venezuela
Venezuelan
5’5”
120 lbs
Black/Grey
Brown
Up To $5 Million
Itriago was charged by indictment in the Southern District of Florida on January 31, 2013, with
conspiracy to import more than five kilograms of cocaine into the United States, in violation of 21 U.S.
Code, Section 959(a)(2) and 21 U.S. Code, Section 963.
Jesus Alfredo Itriago is the former Chief of Counter-narcotics for the Cuerpo de InvestigacionesCientíficas, Penales y Criminálisticas (CICPC) in Venezuela.As a result of his official position in the Venezuelan government, Itriago retained a great deal ofauthority within CICPC Counter-narcotics, which itself had a history of corruption. Accordingly, a DEAinvestigation revealed that because of his drug trafficking associates, and his contacts at the CICPC,airports and maritime ports, Itriago was involved in protecting drug loads departing Venezuela a

Diosdado Cabello Rondón 
https://www.state.gov/wp-content/uploads/2020/03/image1-3.jpg

April 15, 1963
El Furrial, Monagas, Venezuela
Venezuelan
5’10”
250 lbs
Grey
Brown
Up To $10 Million
Cabello Rondón was charged on March 5, 2020 in Southern District of New York federal indictment with conspiracy to commit narco-terrorism, conspiracy to import cocaine, and associated firearms charges in violation of Title 21, United States Code, Sections 960a and 963, and Title 18 United States Code, Sections 924 and 2. 
Diosdado Cabello Rondón is the illegitimate president of Venezuela’s Constituent National Assembly.  Cabello Rondón has also served as President and Vice-President of Venezuela, and is an active member of the Venezuelan Army with the rank of Captain.  Cabello Rondón participated in a corrupt and violent narco-terrorism conspiracy between the Cartel of the Suns, a Venezuelan drug-trafficking organization comprised of high-ranking Venezuelan officials, and the Revolutionary Armed Force

Wilver Villegas Palomino
https://www.state.gov/wp-content/uploads/2020/09/Wilver-Villegas-Palomino.jpg
Carlos El Puerco
October 21, 1981
Curumani, Colombia
Colombian
5’7”

Brown
Brown
Fer Of 50 Million
Villegas Palomino was charged on February 12, 2020, in a Southern District of Texas federal indictment with narco-terrorism, conspiracy to import cocaine, and international cocaine distribution, all in violation of Title 21 U.S.C. 960a, 963, 959a.  According to the indictment, Villegas-Palomino and his codefendants were involved in an ongoing 20-year conspiracy to distribute cocaine from Colombia to the United States knowing or intending to provide pecuniary support to the ELN.
Wilver Villegas Palomino has been a high-ranking member of the Ejército de Liberación Nacional (ELN- National Liberation Army) in Colombia since approximately 2000 as the chief of drug trafficking for the ELN Northeastern War Front (ELN-NEWF).  Villegas is also the commander of the Magdalena area of the ELN-NEWF a

Dario Antonio Usuga David (Captured)
https://www.state.gov/wp-content/uploads/2019/03/Dario-Antonio-Usuga-David-1.jpg
“Mauricio,” “Mao,” “Otoniel”
September 15, 1971
Necocli, Antioquia, Colombia
Colombian
5’ 9”
155 pounds
Black
Brown
Up To $5 Million

Dario Antonio Usuga David is allegedly one of the leaders of El Clan Del Golfo, formerly called Los Urabeños or Autodefensas Gaitanistas de Colombia, a heavily armed and extremely violent Colombian drug cartel comprised of former members of terrorist organizations that did not demobilize as part of the Colombian government’s justice and peace process. The organization uses violence and intimidation to control the narcotics trafficking routes, cocaine processing laboratories, speedboat departure points, and clandestine landing strips. The organization operates in 13 of Colombia’s 32 departments, most of which are in the northwestern part of the country. During a turf war with a rival criminal organization for drug trafficking routes, homic

Luis Antonio Lozada
https://www.state.gov/wp-content/uploads/2019/03/Luis_Antonio_Lozada_225_1.jpg
Carlos Antonio Lozada, Julian Gallo Cubillos
March 24, 1961
Colombia
Colombian


Grey
Brown
Up To $2.5 Million

Luis Antonio Lozada is allegedly a member of the Revolutionary Armed Forces of Colombia (FARC), which is a foreign terrorist organization in Colombia that was established in 1964. The FARC had a Marxist philosophy and is Latin America’s oldest, largest, most capable, and best-equipped insurgency — with perhaps 12,000 fighters and thousands of supporters. The FARC’s declared intent is to overthrow the democratic Colombian government. In addition to its attacks on Colombian military, political, and economic targets, the FARC is deeply involved in narcotics trafficking, kidnapping for ransom, extortion, murder, and other criminal activities.Lozada is allegedly a member of the Estado Mayor and participated in setting and implementing the FARC’s cocaine policies, including directing 

...

Jose Salgueiro-Nevarez
https://www.state.gov/wp-content/uploads/2021/11/20211110_124907-e1636566977145.jpg
“CH,” “CHE,” “El 90,” “Tio,”
December 28, 1966
Mexico
Mexican
5’7”
150-160 lbs
Black
Brown
Up To $5 Million
On February 19, 2020, federal prosecutors in Tucson, Arizona secured an eight-count Superseding Indictment from a federal grand jury in the District of Arizona against fourteen targets representing the senior leadership of the Aureliano Guzman-Loera and SNO factions of the Sinaloa Cartel – including the three Salgueiro-Nevarez brothers.  The three Salgueiro-Nevarez brothers were charged with participating in an international conspiracy to distribute marijuana, heroin, cocaine, and methamphetamine, in violation of federal law.  
WANTED: JOSE SALGUEIRO-NEVAREZ  REWARD OF UP TO $5 MILLION  Noel Salgueiro-Nevarez was arrested in Mexico in 2011, and the control of the SNO eventually moved to Ruperto Salgueiro-Nevarez.  Investigators subsequently linked Ruperto to Aureliano Guzman

Audias Flores-Silva
https://www.state.gov/wp-content/uploads/2021/04/Flores-Silva.jpg
Gabriel Raigosa Plascencia,
November 19, 1980
Michoacan, Mexico
Mexican
5’4”
190 lbs
Brown
Brown
Fer Of Up To $10 Million
Flores-Silva was charged in a federal indictment returned on August 13, 2020, in the U.S. District Court for the District of Columbia.  The indictment charges Flores-Silva with conspiracy to distribute five kilograms or more of cocaine, and one kilogram or more of heroin for importation into the United States, as well as carrying, using, and possessing a firearm in relation to a drug offense. 
Audias Flores-Silva is one of the alleged leaders of the Cartel de Jalisco Nueva Generacion (CJNG), a violent Transnational Criminal Organization operating in Mexico.  The CJNG is assessed to be the most violent drug trafficking organization (DTO) currently operating in Mexico and has the highest cocaine, heroin, and methamphetamine trafficking capacity.  Flores-Silva is very closely aligned 

Up To $5 Million
Juan Jose Esparragoza-Moreno is one of Mexico’s most wanted criminals and allegedly the leader of the Sinaloa Cartel, a drug trafficking organization (DTO). In the 1970’s, Esparragoza-Moreno allegedly founded the Guadalajara Cartel DTO with other Mexican drug kingpins. After the kidnapping, torture, and murder of DEA Special Agent Enrique Camarena, Esparragoza-Moreno was arrested and imprisoned for drug trafficking charges and his alleged participation in the murder of Special Agent Camarena. After Esparragoza-Moreno’s release from prison, he allegedly joined forces with the Juarez Cartel DTO and worked as the operational chief and later became the second-in-command, behind Amado Carrillo Fuentes.
Esparragoza-Moreno is wanted for conspiracy to import a controlled substance, conspiracy to possess with intent to distribute a controlled substance, and importation of a controlled substance. On June 24, 2012, the U.S. Department of Treasury froze the assets of Esparragoza-M

Antonio Indjai–New Target
https://www.state.gov/wp-content/uploads/2021/08/Antonio-Indjai.jpeg
Antonio Injai
January 22, 1955
Guinea-Bissau
Guinea-Bissau


Black
Brown
Up To $5 Million
Indjai was charged in two indictments, an initial indictment, filed on December 12, 2012, and a superseding indictment filed on January 8, 2013. Both indictments were filed in the Southern District of New York, charging Indjai and five others with violations of Title 21, U.S.C. Section 960a, Narcoterrorism Conspiracy, Conspiracy to Import Cocaine in violation of Title 21, U.S.C. Section 959(a) and 960(a)(3).  The first indictment also charged Indjai and his coconspirators in violation of Title 18, U.S.C. Section 2339b, Conspiracy to Provide Material Support to a Foreign Terrorist Organization, and the superseding indictment also charged Indjai and four of his coconspirators with violating Title 18, U.S.C. Section 2332g, Conspiracy to Acquire and Transfer Anti-Aircraft Missiles.  
Antonio Indjai is the fo

Victor Quispe-Palomino
https://www.state.gov/wp-content/uploads/2019/03/Victor_QuispePalomino_225_1.jpg
“Jose,” “Martin,” “Ivan”

Peru
Peruvian


Black
Brown
Up To $5 Million
Quispe-Palomino is allegedly the leader of the Sendero remnants based in the VRAE and oversees all of its illicit activities, including extortion, murder, and drug trafficking. The drug trafficking activities of this faction of Sendero include taxes/extortion payments charged to local drug traffickers in exchange for security of cocaine labs and cocaine shipments made throughout the VRAE. Furthermore, currently Sendero owns several coca plots and cocaine base laboratories in the VRAE. This faction is also notorious for engaging in well-coordinated and extremely violent attacks against Peruvian National Police and Peruvian military personnel.
Victor Quispe-Palomino is allegedly a member of the Sendero Luminoso (also known as the ‘Partido Comunista del Peru’). Founded in 1970 on Communist ideology, the Sendero Lumin